<a href="https://colab.research.google.com/github/Jericho-Ram/FlyRank-Internship-ML/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

**Lane:** content refresh / decline risk — rank existing pages by how likely they are to lose search clicks next month.

Data: the FlyRank warehouse release (`hf://datasets/FlyRank/internship-warehouse`, build v20260703).
Iteration month: `2026-03` (mid-panel). The final month, `2026-06`, is the natural outcome window
of any past→future label, so it stays sealed.

## 0. Setup — token, connection, sanity check

In [ ]:
%pip -q install duckdb

In [ ]:
import os, getpass, duckdb

# Colab Secrets (key panel) -> name it HF_TOKEN. Never paste a token into a cell: this repo is public.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("HF read token: ")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"

def month_rel(m):
    return f"read_parquet('{BASE}/fact_content_daily_performance/month={m}/data_0.parquet')"

MAR = month_rel("2026-03")     # feature window
APR = month_rel("2026-04")     # label window
CLIENTS = f"read_parquet('{BASE}/dim_clients.parquet')"

print("connected")

connected


In [ ]:
# Near-free: COUNT and MIN/MAX over Parquet touch metadata, not rows.
from huggingface_hub import list_repo_files
files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=HF_TOKEN)
print(len(files))
for f in files[:40]:
    print(f)

24
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/m

In [ ]:
# Real column names, before writing any query that assumes them.
schema = con.sql(f"SELECT * FROM {MAR} LIMIT 1").df()
print(list(schema.columns))

def pick(*candidates):
    """Resolve the first candidate that actually exists. Fail loudly, not silently."""
    cols = set(schema.columns)
    for c in candidates:
        if c in cols:
            return c
    raise KeyError(f"none of {candidates} in table; columns are {sorted(cols)}")

IMPR  = pick("gsc_impressions", "impressions")
CLICK = pick("gsc_clicks", "clicks")
POS   = pick("gsc_avg_position", "avg_position")
GA4OK = pick("ga4_data_available")
print(IMPR, CLICK, POS, GA4OK)

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
gsc_impressions gsc_clicks gsc_avg_position ga4_data_available


## 1. The contract, in plain words

**One row means:** one content item on one day for one client — `report_date` × `client_id` × `content_id`
in `fact_content_daily_performance`. Verified in query 1 below.

**Tables I use:** `fact_content_daily_performance_sample` for daily search and engagement counts,
`dim_clients` for each client's history start dates. I do not touch `fact_content_query_90d` —
its fixed 90-day window overlaps my label window, so nothing in it is a legal feature without
window surgery I don't need yet.

**Time window:** features from `2026-03` only. The label comes from `2026-04` only. Prior window →
future window, no overlap, by construction.

**What I rank:** decline risk. `is_declining = clicks in 2026-04 < clicks in 2026-03`, per content item.
This is a defined rule on an observed outcome, not a judgement call — and it lives entirely after
the feature window closes.

**Excluded on purpose:** every column dated inside or after `2026-04`. That includes April clicks,
the column my label is computed from. It is the label's raw material, so it can never be a feature —
I prove what happens if it is in section 4.

## 2. Three facts, three queries

Each claim above gets its own query. A contract line without a query next to it is a guess.

In [ ]:
MONTH = "2026-03"

# Q1 — GRAIN: one row really is one report_date x client_hash_id x content_hash_id.
# Zero rows back means the grain holds.
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {MAR}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


In [ ]:
# Q2 — SIZE AND SPAN of my slice: how many rows, how many items, which dates.
con.sql(f"""
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT content_hash_id) AS content_items,
           COUNT(DISTINCT client_hash_id) AS clients,
           MIN(report_date) AS first_day,
           MAX(report_date) AS last_day
    FROM {MAR}
""").df()

,rows,content_items,clients,first_day,last_day
0,9841378,331437,55,2026-03-01,2026-03-31


In [ ]:
# Q3 — AVAILABILITY: rows before a client's ga4_data_start are zero-FILLED, not zero.
# Filter on the flag and show how many rows survive.
con.sql(f"""
    SELECT COUNT(*) AS all_rows,
           COUNT(*) FILTER (WHERE {GA4OK} IS TRUE) AS ga4_usable_rows,
           ROUND(100.0 * COUNT(*) FILTER (WHERE {GA4OK} IS TRUE) / COUNT(*), 1) AS pct_surviving
    FROM {MAR}
""").df()
# of 9,841,378 rows in March, 413,966 (4.2%) have ga4_data_available IS TRUE; the rest carry zero-filled GA4 columns that would silently drag any engagement average toward zero

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_rows,ga4_usable_rows,pct_surviving
0,9841378,413966,4.2


## 3. Five features, and when each is knowable

Built from `2026-03` alone. One line each, in the form the assignment asks for.

| # | Feature | Knowable at the decision moment because… |
|---|---|---|
| 1 | `impressions_mar` | it counts March impressions only, and I decide on 1 April |
| 2 | `clicks_mar` | same window — March clicks are closed and countable before April opens |
| 3 | `ctr_mar` | a ratio of two March-only columns; no April input anywhere in it |
| 4 | `avg_position_mar` | March's mean position, computed on days that actually had impressions |
| 5 | `days_with_impressions_mar` | a count of March days; consistency of exposure, fully in the past |

Nothing here reads a date after 2026-03-31. That is the whole test.

In [ ]:
features = con.sql(f"""
    SELECT client_hash_id,
           content_hash_id,
           SUM({IMPR})                                        AS impressions_mar,
           SUM({CLICK})                                       AS clicks_mar,
           100.0 * SUM({CLICK}) / NULLIF(SUM({IMPR}), 0)      AS ctr_mar,
           AVG(CASE WHEN {POS} > 0 THEN {POS} END)            AS avg_position_mar,
           COUNT(*) FILTER (WHERE {IMPR} > 0)                 AS days_with_impressions_mar
    FROM {MAR}
    GROUP BY 1, 2
    HAVING SUM({IMPR}) > 0
""").df()

print(features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 7)


,client_hash_id,content_hash_id,impressions_mar,clicks_mar,ctr_mar,avg_position_mar,days_with_impressions_mar
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,0.000000,4.888929,24
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,602.0,4.0,0.664452,4.428747,29
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,810.0,1.0,0.123457,4.866123,29
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,82.0,0.0,0.000000,10.100347,27
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,1858.0,6.0,0.322928,1.854929,30


In [ ]:
# The label — April only. Built after the feature window closes, never before.
label = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM({CLICK}) AS clicks_apr
        FROM {APR}
            GROUP BY 1, 2
            """).df()

df = features.merge(label, on=["client_hash_id", "content_hash_id"], how="inner")
df["is_declining"] = (df["clicks_apr"] < df["clicks_mar"]).astype(int)
print("rows:", len(df))
print("base rate (declining):", round(df["is_declining"].mean(), 3))
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 176737
base rate (declining): 0.255


,client_hash_id,content_hash_id,impressions_mar,clicks_mar,ctr_mar,avg_position_mar,days_with_impressions_mar,clicks_apr,is_declining
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,0.000000,4.888929,24,0.0,0
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,602.0,4.0,0.664452,4.428747,29,1.0,1
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,810.0,1.0,0.123457,4.866123,29,0.0,1
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,82.0,0.0,0.000000,10.100347,27,0.0,0
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,1858.0,6.0,0.322928,1.854929,30,1.0,1


## 4. The trap — spring it on purpose

I add one label-derived column, `clicks_apr`, as a feature. The label is computed from it, so the
model gets to read the answer. Watch the score run toward perfect, then delete the column and keep
the honest number.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
import numpy as np

HONEST = ["impressions_mar", "clicks_mar", "ctr_mar", "avg_position_mar", "days_with_impressions_mar"]
LEAKY  = HONEST + ["clicks_apr"]  # deliberate leak

work = df.dropna(subset=HONEST + ["clicks_apr"]).copy()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
tr, te = next(splitter.split(work, groups=work["client_hash_id"]))

# Whole clients held out, not random rows: the honest question is "does it work on a client it never saw?"
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
tr, te = next(splitter.split(work, groups=work["client_hash_id"]))

def quick_auc(cols):
    m = RandomForestClassifier(n_estimators=120, random_state=0, n_jobs=-1)
    m.fit(work.iloc[tr][cols], work.iloc[tr]["is_declining"])
    p = m.predict_proba(work.iloc[te][cols])[:, 1]
    return roc_auc_score(work.iloc[te]["is_declining"], p)

auc_leaky  = quick_auc(LEAKY)
auc_honest = quick_auc(HONEST)

print(f"WITH the label-derived column : ROC AUC {auc_leaky:.3f}   <- too good, and worthless")
print(f"WITHOUT it (honest)           : ROC AUC {auc_honest:.3f}")
print(f"base rate                     : {work['is_declining'].mean():.3f}")
print(f"the confession                : {auc_leaky - auc_honest:+.3f}")

WITH the label-derived column : ROC AUC 1.000   <- too good, and worthless
WITHOUT it (honest)           : ROC AUC 0.949
base rate                     : 0.257
the confession                : +0.051


In [ ]:
# Delete it. The honest feature set is what carries forward.
del LEAKY
FINAL_FEATURES = HONEST
print("kept:", FINAL_FEATURES)

kept: ['impressions_mar', 'clicks_mar', 'ctr_mar', 'avg_position_mar', 'days_with_impressions_mar']


In [ ]:
NO_CLICKS = ["impressions_mar", "ctr_mar", "avg_position_mar", "days_with_impressions_mar"]
print(f"without clicks_mar: ROC AUC {quick_auc(NO_CLICKS):.3f}")
print("declining rate when clicks_mar == 0:",
      round(work.loc[work.clicks_mar == 0, "is_declining"].mean(), 3))

without clicks_mar: ROC AUC 0.949
declining rate when clicks_mar == 0: 0.0


**What happened:** the leaky run climbs toward 1.0 because `clicks_apr` is half of the comparison
that defines `is_declining` — the model isn't predicting, it's reading. The honest number is the one
I report, and it is the one a buyer can act on.

*Fill in the actual numbers here in one sentence after you run it — the drop is your finding.*

With clicks_apr as a feature the model scored ROC AUC 1.000 against a base rate of 0.257; removing it left 0.949. Dropping clicks_mar too changed nothing (0.949), which is the more interesting result: pages with zero March clicks are declining 0.0% of the time by construction, so the label leaks through the whole feature set, not through any one column.

## 5. One named limitation of my slice

**Clients don't share a start line.** `dim_clients.gsc_data_start` differs per client, and rows before
a client's `ga4_data_start` come back zero-filled with `ga4_data_available = FALSE`. So a global
calendar month is not the same amount of history for every client — for some it's a full month,
for others it's the first weeks they existed in the data. Any per-client aggregate I compute without
checking this is measuring onboarding date as much as content performance. Shown below.

A second limit, found by testing: is_declining is defined as clicks_apr < clicks_mar, so pages with zero March clicks are labelled not-declining 0.0% of the time by construction. Removing clicks_mar from the features left ROC AUC unchanged at 0.949 — the leak is in the label's definition, not in any one column, and it would need a label redesign to fix.

In [ ]:
con.sql(f"""
    SELECT MIN(gsc_data_start) AS earliest_client_start,
           MAX(gsc_data_start) AS latest_client_start,
           COUNT(*)            AS clients
    FROM {CLIENTS}
""").df()

,earliest_client_start,latest_client_start,clients
0,2025-01-27,2026-06-02,104


In [ ]:
# Same point, inside my month: how uneven are the clients I actually modelled?
per_client = df.groupby("client_hash_id").size().sort_values()
print("content items per client — min:", per_client.iloc[0], "| max:", per_client.iloc[-1])
print("top client's share of all rows:", round(100 * per_client.iloc[-1] / len(df), 1), "%")

content items per client — min: 2 | max: 27425
top client's share of all rows: 15.5 %


## Self-check

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.